# Reasoning Memory

The previous notebooks introduced short-term memory (conversation history) and long-term memory (entities, preferences, facts). This notebook covers the third memory type: **reasoning memory** — traces of past agent executions that enable the agent to learn from experience.

When an agent completes a task, you can record a trace capturing what it did, which tools it called, what the outcome was, and whether it succeeded. Later, when the agent encounters a similar task, it can retrieve those traces to reuse successful strategies or avoid approaches that failed.

## What you will learn

- How to record agent execution traces with `record_agent_trace()`
- How to find similar past traces with `get_similar_traces()`
- How to view tool usage statistics with `get_tool_stats()`

***

Load the environment variables and import the required modules.

In [ ]:
import sys
sys.path.insert(0, '../shared')

from pydantic import SecretStr

from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.integrations.microsoft_agent import (
    Neo4jMicrosoftMemory,
    record_agent_trace,
    get_similar_traces,
)

from azure_embedder import get_memory_embedder
from config import Neo4jConfig

## Configure and Connect

In [ ]:
neo4j_config = Neo4jConfig()

settings = MemorySettings(
    neo4j={
        "uri": neo4j_config.uri,
        "username": neo4j_config.username,
        "password": SecretStr(neo4j_config.password),
    },
)

memory_client = MemoryClient(settings, embedder=get_memory_embedder())
await memory_client.__aenter__()
print("Connected to Neo4j")

## Create Memory

In [ ]:
memory = Neo4jMicrosoftMemory.from_memory_client(
    memory_client=memory_client,
    session_id="reasoning-demo",
    include_short_term=True,
    include_long_term=True,
    include_reasoning=True,
)
print("Memory created with reasoning enabled")

## Record an Agent Trace

Use `record_agent_trace()` to capture a completed agent execution. Each trace records:

- **task** — what the agent was trying to accomplish
- **tool_calls** — which tools were invoked, with arguments, results, status, and duration
- **outcome** — the final result
- **success** — whether the task completed successfully

The trace gets a vector embedding of the task description, enabling semantic search later.

In [ ]:
trace = await record_agent_trace(
    memory=memory,
    messages=[],
    task="Find risk factors related to supply chain disruptions in Apple's 10-K filing",
    tool_calls=[
        {
            "name": "search_knowledge",
            "arguments": {"query": "Apple supply chain risk factors"},
            "result": ["Component supply constraints", "Single-source supplier risk", "Geopolitical disruptions"],
            "status": "success",
            "duration_ms": 180,
        },
        {
            "name": "search_memory",
            "arguments": {"query": "user interest in supply chain"},
            "result": ["User previously expressed concern about supply chain risks"],
            "status": "success",
            "duration_ms": 95,
        },
    ],
    outcome="Identified 3 key supply chain risk factors from Apple's 10-K and connected them to user's stated interest",
    success=True,
)
print(f"Recorded trace: {trace.id}")
print(f"Task: {trace.task}")
print(f"Steps: {len(trace.steps)}")
print(f"Success: {trace.success}")

Record a second trace — this one a failed attempt — so the agent can learn what doesn't work.

In [ ]:
failed_trace = await record_agent_trace(
    memory=memory,
    messages=[],
    task="Find competitive analysis between Apple and Microsoft from SEC filings",
    tool_calls=[
        {
            "name": "search_knowledge",
            "arguments": {"query": "Apple vs Microsoft competition"},
            "result": [],
            "status": "success",
            "duration_ms": 210,
        },
    ],
    outcome="No competitive analysis found — 10-K filings discuss risks independently, not as head-to-head comparisons",
    success=False,
)
print(f"Recorded trace: {failed_trace.id}")
print(f"Task: {failed_trace.task}")
print(f"Success: {failed_trace.success}")

## Find Similar Traces

Use `get_similar_traces()` to search for reasoning traces from similar past tasks. The search uses the vector embedding of the task description to find semantically similar traces — so the agent can learn from its own experience.

In [ ]:
traces = await get_similar_traces(
    memory=memory,
    task="What are the supply chain risks in Microsoft's annual report?",
    limit=3,
)

print(f"Found {len(traces)} similar traces:\n")
for t in traces:
    print(f"Task: {t.task}")
    print(f"Outcome: {t.outcome}")
    print(f"Success: {t.success}")
    print(f"Steps: {len(t.steps)}")
    print()

> The supply chain query matched the successful trace from earlier. An agent could use this to replicate the same tool-calling strategy — searching the knowledge graph first, then checking user preferences — rather than starting from scratch.

## View Tool Statistics

Reasoning memory tracks aggregate statistics about tool usage across all recorded traces. These help identify which tools are reliable and fast vs. which fail frequently or are slow.

In [ ]:
stats = await memory_client.reasoning.get_tool_stats()

if stats:
    print("Tool Statistics:\n")
    for tool in stats:
        print(f"  {tool.name}:")
        print(f"    Calls: {tool.total_calls} ({tool.successful_calls} success, {tool.failed_calls} failed)")
        print(f"    Success rate: {tool.success_rate:.0%}")
        if tool.avg_duration_ms:
            print(f"    Avg duration: {tool.avg_duration_ms:.0f}ms")
        print()
else:
    print("No tool statistics recorded yet.")

## Cleanup

In [ ]:
await memory_client.__aexit__(None, None, None)
print("Connection closed")

## Summary

Reasoning memory records agent execution traces — tasks, tool calls, outcomes, and success/failure status. These traces are stored with vector embeddings so the agent can find similar past tasks using semantic search. Combined with the automatic context injection from Notebook 01 (`include_reasoning=True`), the agent can learn from past successes and failures without any explicit tool calls.

***

[View the complete code](../financial_data_load/solution_srcs/07_04_reasoning_memory.py)

[Continue to Lab 8 - Building a Knowledge Graph](../Lab_8_Knowledge_Graph)